In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import os

# ==========================================================================
# 1. DOSYA YOLLARINI TANIMLAMA
# ==========================================================================
train_path = '/content/drive/MyDrive/data/ai-ready-data/train_scaled.csv'
test_path  = '/content/drive/MyDrive/data/ai-ready-data/test_scaled.csv'
shap_csv_yolu = '/content/drive/MyDrive/data/ai-ready-data/shap_outputs_2/06_shap_importance_summary.csv'
atilacaklar_csv_yolu = '/content/drive/MyDrive/data/ai-ready-data/shap_outputs_2/07_atilacak_ozellikler_shap_degerine_gore_1.csv'

# ==========================================================================
# 2. VERİLERİ YÜKLEME
# ==========================================================================
print("📂 Veri setleri ve SHAP analiz raporu yükleniyor, lütfen bekleyin...")
train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)
shap_df  = pd.read_csv(shap_csv_yolu)
print("✅ Yükleme tamamlandı.\n")

# ==========================================================================
# 3. MATEMATİKSEL SHAP EŞİK DEĞERİ VE ÖZELLİK SEÇİMİ
# ==========================================================================
# Tüm özelliklerin global SHAP ortalamasını "Kesme Noktası" (Threshold) yapıyoruz
threshold =0.01
print(f"📊 MATEMATİKSEL EŞİK DEĞERİ (SHAP ORTALAMASI): {threshold:.6f}")

# Eşiğin altında kalanları atılacak, üstünde kalanları seçilecek olarak ayırıyoruz
kalan_df = shap_df[shap_df['global_mean_abs_shap'] > threshold].copy()
atilacak_df = shap_df[shap_df['global_mean_abs_shap'] <= threshold].copy()

kalan_ozellikler = kalan_df['feature'].tolist()
atilacak_ozellikler = atilacak_df['feature'].tolist()

print(f"💎 Seçilen (Kaliteli) Özellik Sayısı: {len(kalan_ozellikler)}")
print(f"🗑️ Atılacak (Gürültü) Özellik Sayısı: {len(atilacak_ozellikler)}\n")

# Atılacakları Drive'a temiz bir CSV olarak kaydetme
atilacak_df[['feature', 'global_mean_abs_shap']].to_csv(atilacaklar_csv_yolu, index=False)
print(f"💾 Atılacak özellikler raporu kaydedildi: {atilacaklar_csv_yolu}\n")

# ==========================================================================
# 4. GÜRÜLTÜ ÖZELLİKLERİ VERİ SETİNDEN TAMAMEN SİLME
# ==========================================================================
# errors='ignore' parametresi, zaten silinmiş bir sütunu tekrar silmeye çalışırsa kodu çökertmez
train_df = train_df.drop(columns=atilacak_ozellikler, errors='ignore')
test_df  = test_df.drop(columns=atilacak_ozellikler, errors='ignore')

# ==========================================================================
# 5. MODEL İÇİN X (ÖZELLİK) VE y (ETİKET) AYRIMI
# ==========================================================================
X_train = train_df.drop('Label', axis=1).values
y_train = train_df['Label'].values

X_test = test_df.drop('Label', axis=1).values
y_test = test_df['Label'].values

num_classes = len(np.unique(y_train))

# ==========================================================================
# 6. 1D-CNN İÇİN GÜVENLİ YENİDEN BOYUTLANDIRMA (RESHAPE)
# ==========================================================================
# Label kolonu hariç, geriye kalan net özellik sayısını alıyoruz
feature_count = train_df.shape[1] - 1

# Numpy'ı (Örnek Sayısı, Özellik Sayısı, 1) formatına zorluyoruz
X_train_cnn = X_train.reshape(-1, feature_count, 1)
X_test_cnn  = X_test.reshape(-1, feature_count, 1)

# ==========================================================================
# SONUÇ ÖZETİ
# ==========================================================================
print("==========================================================================")
print("🚀 --- 1D-CNN VERİ HAZIRLIK SÜRECİ BAŞARIYLA TAMAMLANDI --- 🚀")
print("==========================================================================")
print(f"🎯 Hedef Sınıf Sayısı     : {num_classes}")
print(f"📉 Orijinal Özellik Sayısı: {len(shap_df)}")
print(f"📈 Modele Giren Özellik   : {feature_count}")
print(f"📦 Yeni X_train Boyutu    : {X_train_cnn.shape}")
print(f"📦 Yeni X_test Boyutu     : {X_test_cnn.shape}")
print("==========================================================================")

📂 Veri setleri ve SHAP analiz raporu yükleniyor, lütfen bekleyin...
✅ Yükleme tamamlandı.

📊 MATEMATİKSEL EŞİK DEĞERİ (SHAP ORTALAMASI): 0.010000
💎 Seçilen (Kaliteli) Özellik Sayısı: 37
🗑️ Atılacak (Gürültü) Özellik Sayısı: 27

💾 Atılacak özellikler raporu kaydedildi: /content/drive/MyDrive/data/ai-ready-data/shap_outputs_2/07_atilacak_ozellikler_shap_degerine_gore_1.csv

🚀 --- 1D-CNN VERİ HAZIRLIK SÜRECİ BAŞARIYLA TAMAMLANDI --- 🚀
🎯 Hedef Sınıf Sayısı     : 8
📉 Orijinal Özellik Sayısı: 64
📈 Modele Giren Özellik   : 37
📦 Yeni X_train Boyutu    : (1765620, 37, 1)
📦 Yeni X_test Boyutu     : (756695, 37, 1)


In [3]:
# ==========================================================================
# 7. TEMİZLENMİŞ (ÖZELLİK SEÇİMİ YAPILMIŞ) VERİ SETLERİNİ KAYDETME
# ==========================================================================
# Kaydedilecek yeni temiz veri dosyalarının yollarını tanımlıyoruz
clean_train_path = '/content/drive/MyDrive/data/ai-ready-data/train_scaled_shap.csv'
clean_test_path  = '/content/drive/MyDrive/data/ai-ready-data/scaled_shap.csv'

print("💾 Mutlak SHAP değerlerine göre filtrelenmiş  veri setleri kaydediliyor...")

# train_df ve test_df içerisinde sadece SHAP eşiğini geçen özellikler ve 'Label' kolonu kaldı.
# Index'i kaydetmemek için index=False kullanıyoruz.
train_df.to_csv(clean_train_path, index=False)
test_df.to_csv(clean_test_path, index=False)

print("==========================================================================")
print(f"✅ Temizlenmiş Eğitim Verisi Kaydedildi: {clean_train_path}")
print(f"✅ Temizlenmiş Test Verisi Kaydedildi  : {clean_test_path}")
print("==========================================================================")

💾 Mutlak SHAP değerlerine göre filtrelenmiş  veri setleri kaydediliyor...
✅ Temizlenmiş Eğitim Verisi Kaydedildi: /content/drive/MyDrive/data/ai-ready-data/train_scaled_shap.csv
✅ Temizlenmiş Test Verisi Kaydedildi  : /content/drive/MyDrive/data/ai-ready-data/scaled_shap.csv
